In [ ]:
import importlib.util, json, platform, shutil, subprocess, sys, time
from pathlib import Path
report = {"python": sys.version, "platform": platform.platform(), "packages": {}, "jax_devices": [], "torch_xla_devices": [], "errors": {}}
print("TPU V5E8 EXPLICIT HARDWARE PROBE")
for pkg in ("jax", "jaxlib", "torch", "torch_xla"):
    report["packages"][pkg] = importlib.util.find_spec(pkg) is not None
try:
    import jax
    report["jax_version"] = jax.__version__
    report["jax_devices"] = [{"platform": d.platform, "kind": getattr(d,"device_kind",None), "id": d.id} for d in jax.devices()]
    print("JAX", jax.__version__, "DEVICES", json.dumps(report["jax_devices"]), flush=True)
except Exception as e: report["errors"]["jax"] = repr(e)
try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    report["torch_xla_devices"] = xm.get_xla_supported_devices()
    report["torch_xla_version"] = getattr(torch_xla,"__version__","unknown")
except Exception as e: report["errors"]["torch_xla"] = repr(e)
for cmd in (["nvidia-smi"], ["df","-h","/kaggle/working"], ["free","-h"]):
    try:
        p=subprocess.run(cmd,text=True,capture_output=True,timeout=15)
        report["system_"+cmd[0]] = (p.stdout or p.stderr)[-6000:]
        print("SYSTEM",cmd[0],(p.stdout or p.stderr)[-1200:],flush=True)
    except Exception as e: report["errors"][cmd[0]]=repr(e)
report["tpu_detected"] = any(d.get("platform","").lower()=="tpu" for d in report["jax_devices"]) or bool(report["torch_xla_devices"])
Path("/kaggle/working/tpu_v5e8_probe.json").write_text(json.dumps(report,indent=2))
print("TPU_V5E8_PROBE",json.dumps(report),flush=True)
if not report["tpu_detected"]: print("MODEL_TESTS_SKIPPED_NO_TPU", flush=True)
